# 🚲 HLBT Baku - Bike: Velosiped İcarəsi Proqnozu

**Məqsəd:** Hava şəraiti, vaxt və digər faktorlara əsasən saatlıq velosiped icarəsini proqnozlaşdırmaq.

**Metrika:** MAE (Mean Absolute Error) — nə qədər aşağı olsa, bir o qədər yaxşıdır.

**Pipeline:**
1. Kitabxanaları yüklə
2. Datanı oxu
3. EDA (Exploratory Data Analysis)
4. Data təmizliyi
5. Feature Engineering
6. Model qur
7. Submission faylı hazırla

## 📦 1. Kitabxanaları yüklə

In [ ]:
import pandas as pd          # Cədvəl (DataFrame) işləmək üçün
import numpy as np           # Riyazi əməliyyatlar üçün

# Vizualizasiya kitabxanaları
import matplotlib.pyplot as plt   # Qrafik çəkmək üçün
import seaborn as sns             # Daha gözəl qrafiklər üçün

# Maşın öyrənməsi kitabxanaları
from sklearn.ensemble import RandomForestRegressor   # Random Forest modeli
from sklearn.model_selection import cross_val_score  # Cross-validation
from sklearn.metrics import mean_absolute_error      # MAE hesablamaq üçün

# Xəbərdarlıqları gizlət (kod daha səliqəli görünsün)
import warnings
warnings.filterwarnings('ignore')

# Qrafik ölçüsünü ayarla
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('✅ Bütün kitabxanalar uğurla yükləndi!')

ModuleNotFoundError: No module named 'matplotlib'

## 📂 2. Datanı oxu

In [ ]:
# CSV fayllarını pandas DataFrame-ə oxu
# Fayllar bu notebook ilə eyni qovluqda olmalıdır
train = pd.read_csv('train.csv')            # Öyrətmə datası
test  = pd.read_csv('test.csv')             # Test datası (count yoxdur)
sub   = pd.read_csv('sample_submission.csv') # Göndərmə nümunəsi

print(f'Train ölçüsü : {train.shape}')   # (sətir sayı, sütun sayı)
print(f'Test ölçüsü  : {test.shape}')
print(f'Submission   : {sub.shape}')

In [ ]:
# İlk 5 sətirə bax — datanın necə göründüyünü anla
train.head()

In [ ]:
# Hər sütunun tipi və boş dəyər sayı
# object = mətn, int64 = tam ədəd, float64 = onluq ədəd
train.info()

In [ ]:
# Statistik icmal: min, max, ortalama, standart kənara çıxma
train.describe().round(2)

## 🔍 3. EDA — Datanı kəşf et

In [ ]:
# Boş dəyər yoxlaması
# isnull() — hər hücrə boşdurmu? True/False qaytarır
# sum() — True-ları say
print('Boş dəyərlər:')
print(train.isnull().sum())

In [ ]:
# Mənfi count dəyərləri — bunlar data xətasıdır
neg_mask = train['count'] < 0
print(f'Mənfi count sayı: {neg_mask.sum()}')
print(f'Ən kiçik dəyər : {train["count"].min()}')

# Mənfi olan sətirləri göstər
train[neg_mask].head()

In [ ]:
# datetime sütununu tarix formatına çevir (hələlik EDA üçün)
train['datetime'] = pd.to_datetime(train['datetime'])

# Saatlıq ortalama icarə
by_hour = train.groupby(train['datetime'].dt.hour)['count'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Saata görə icarə
axes[0].bar(by_hour.index, by_hour.values, color='steelblue', alpha=0.8)
axes[0].set_title('Saata görə ortalama icarə')
axes[0].set_xlabel('Saat')
axes[0].set_ylabel('Ortalama count')
axes[0].axvspan(7, 9, alpha=0.2, color='red', label='Rush hour (səhər)')
axes[0].axvspan(17, 19, alpha=0.2, color='orange', label='Rush hour (axşam)')
axes[0].legend()

# Sağ: Count paylanması
axes[1].hist(train['count'], bins=40, color='green', alpha=0.7, edgecolor='white')
axes[1].set_title('Count dəyərinin paylanması (sağa əyri)')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Tezlik')

plt.tight_layout()
plt.show()

In [ ]:
# Korrelyasiya matrisi — hansı feature count ilə ən çox bağlıdır?
# Korrelyasiya: +1 = mükəmməl müsbət, -1 = mükəmməl mənfi, 0 = əlaqə yoxdur

num_cols = ['season','holiday','workingday','weather','temp',
            'atemp','humidity','windspeed','count']

corr = train[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,      # Dəyərləri göstər
    fmt='.2f',       # 2 onluq rəqəm
    cmap='RdYlGn',   # Qırmızı (mənfi) → Yaşıl (müsbət)
    center=0,        # 0 mərkəzdə
    square=True
)
plt.title('Korrelyasiya Matrisi')
plt.tight_layout()
plt.show()

print('Count ilə korrelyasiya (güclüdən zəifə):')
print(corr['count'].drop('count').sort_values(ascending=False))

In [ ]:
# Mövsüm, hava, həftə günü üzrə vizual analiz
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Mövsüm
season_map = {1: 'Yaz', 2: 'Yay', 3: 'Payız', 4: 'Qış'}
train['season_name'] = train['season'].map(season_map)
train.groupby('season_name')['count'].mean().plot(kind='bar', ax=axes[0],
    color=['green','orange','brown','skyblue'], rot=0)
axes[0].set_title('Mövsümə görə ortalama icarə')
axes[0].set_xlabel('')

# Hava
weather_map = {1: 'Aydın', 2: 'Dumanlı', 3: 'Yağışlı', 4: 'Güclü yağış'}
train['weather_name'] = train['weather'].map(weather_map)
train.groupby('weather_name')['count'].mean().plot(kind='bar', ax=axes[1],
    color='steelblue', rot=15)
axes[1].set_title('Hava şəraitinə görə icarə')
axes[1].set_xlabel('')

# Temp vs Count scatter
axes[2].scatter(train['temp'], train['count'], alpha=0.3, s=10, color='coral')
axes[2].set_title('Temperatur vs Count')
axes[2].set_xlabel('Temperatur (°C)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 🧹 4. Data Təmizliyi

In [ ]:
# Mənfi count dəyərlərini sil — bunlar data xətasıdır
before = len(train)

train = train[train['count'] >= 0].copy()
# .copy() — orijinal datanı dəyişdirməmək üçün yeni kopya yaradır

after = len(train)
print(f'Silindən əvvəl: {before} sətir')
print(f'Silindikdən sonra: {after} sətir')
print(f'Silindi: {before - after} sətir')

## ⚙️ 5. Feature Engineering — Yeni sütunlar yarat

**Feature Engineering** — mövcud məlumatdan yeni, daha faydalı sütunlar yaratmaq deməkdir.
Məsələn, `datetime`-dən `hour` çıxarmaq modelin vaxt nümunəsini öyrənməsinə kömək edir.

In [ ]:
def add_features(df):
    """
    Hər iki dataset üçün (train və test) eyni feature-ləri əlavə edir.
    Funksiya yazmağın üstünlüyü: kodu iki dəfə yazmırıq, səhv riski azalır.
    """
    df = df.copy()  # Orijinalı qorumaq üçün

    # datetime sütununu pandas tarix formatına çevir
    df['datetime'] = pd.to_datetime(df['datetime'])

    # Tarixdən yeni sütunlar çıxar
    df['hour']    = df['datetime'].dt.hour       # 0-23: günün saatı
    df['month']   = df['datetime'].dt.month      # 1-12: il ayı
    df['weekday'] = df['datetime'].dt.dayofweek  # 0=Bazar ertəsi, 6=Bazar
    df['year']    = df['datetime'].dt.year       # 2011 və ya 2012

    # Rush hour: səhər (7-9) və axşam (17-19) pik saatları
    # .isin() — dəyər həmin siyahıdadırmı? True/False qaytarır
    # .astype(int) — True→1, False→0 çevirir
    df['rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19]).astype(int)

    # Gecə saatları: adətən çox az icarə var
    df['night'] = df['hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)

    # İş günü + Rush hour birləşməsi
    # İş günündə rush hour → çox icarə
    # Həftə sonunda həmin saatlar → az icarə
    df['workingday_rush'] = df['workingday'] * df['rush_hour']

    return df


# Hər iki dataset-ə tətbiq et
train = add_features(train)
test  = add_features(test)

print('Yeni sütunlar əlavə edildi!')
print('Train sütunları:', list(train.columns))

In [ ]:
# Log Transform — count-u logaritmik skala ilə çevir
#
# Niyə lazımdır?
# - count sağa əyridir (çox kiçik, az böyük dəyər var)
# - Log transform bu əyriliyi düzəldir
# - MAE metrikası log scaleda daha yaxşı işləyir
#
# np.log1p(x) = log(x+1)  →  0 dəyərini qorumaq üçün +1 əlavə edirik
# (log(0) = -sonsuz olduğundan xəta verər)

train['log_count'] = np.log1p(train['count'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(train['count'], bins=40, color='coral', alpha=0.8, edgecolor='white')
axes[0].set_title('Orijinal count (sağa əyri)')
axes[0].set_xlabel('Count')

axes[1].hist(train['log_count'], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
axes[1].set_title('log(count+1) — normal paylanmaya yaxın')
axes[1].set_xlabel('log(Count+1)')

plt.tight_layout()
plt.show()

## 🤖 6. Model Qur

In [ ]:
# Modelin istifadə edəcəyi sütunları seç
# 'casual', 'registered', 'count', 'log_count' — bunlar TARGET-dır, feature deyil!
# 'datetime', 'season_name', 'weather_name' — model üçün lazımsız

features = [
    'season', 'holiday', 'workingday', 'weather',  # Kateqoriyalı
    'temp', 'atemp', 'humidity', 'windspeed',       # Ədədi
    'hour', 'month', 'weekday', 'year',             # Zamanla bağlı (yeni)
    'rush_hour', 'night', 'workingday_rush'         # Düzəldilmiş (yeni)
]

# X = featurelər (giriş), y = target (çıxış)
X      = train[features]
y      = train['log_count']   # Log-u öyrənir, sonra geri çeviririk
X_test = test[features]

print(f'X ölçüsü    : {X.shape}')
print(f'y ölçüsü    : {y.shape}')
print(f'X_test ölçüsü: {X_test.shape}')

In [ ]:
# Cross-Validation (Çarpaz Doğrulama)
#
# Niyə lazımdır?
# - Modeli bütün data ilə öyrətsən, skoru necə yoxlayacaqsan?
# - CV datanı 5 hissəyə bölür, növbə ilə 1-ini test kimi istifadə edir
# - 5 fərqli skor alırsan → ortalamaları götür = etibarlı qiymət
#
# cv=5 → 5-fold cross validation
# n_jobs=-1 → bütün CPU nüvələrini işlət (sürətli)

rf = RandomForestRegressor(
    n_estimators=200,   # 200 qərar ağacı
    random_state=42,    # Nəticə hər dəfə eyni olsun
    n_jobs=-1           # Paralel işlə
)

print('Cross-validation başlayır... (1-2 dəqiqə çəkə bilər)')

# scoring='neg_mean_absolute_error' — sklearn mənfi dəyər qaytarır
# Biz -1 ilə vururuq ki, müsbət olsun
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='neg_mean_absolute_error')
cv_mae = -cv_scores.mean()

print(f'\n5 CV skoru   : {(-cv_scores).round(4)}')
print(f'Ortalama MAE : {cv_mae:.4f} (log scale)')
print(f'\nBu o deməkdir ki, model ortalama ~{np.expm1(cv_mae):.1f} vahid yanılır')

In [ ]:
# Tam datanı istifadə edərək final modeli öyrət
# CV yalnız skor qiymətləndirmək üçün idi
# İndi bütün train datası ilə öyrədirik ki, daha güclü olsun

print('Final model öyrədilir...')
rf.fit(X, y)
print('✅ Model hazırdır!')

In [ ]:
# Feature Importance — hansı sütun modeli daha çox istiqamətləndirir?
importances = pd.Series(rf.feature_importances_, index=features)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
colors = ['steelblue' if v > 0.05 else 'lightsteelblue' for v in importances.values]
importances.plot(kind='barh', color=colors)
plt.title('Feature Importance — Hansı sütun daha vacibdir?')
plt.xlabel('Əhəmiyyət dərəcəsi')
plt.axvline(x=0.05, color='red', linestyle='--', alpha=0.5, label='5% hədd')
plt.legend()
plt.tight_layout()
plt.show()

print('\nƏn vacib 5 feature:')
print(importances.sort_values(ascending=False).head().round(4))

## 📤 7. Proqnoz et və Submission faylı hazırla

In [ ]:
# Test datası üçün proqnoz
preds_log = rf.predict(X_test)

# Log scaleni geri çevir: expm1(x) = e^x - 1
# Bu log1p()-in tam əksidir
preds = np.expm1(preds_log)

# Tam ədədə çevir (icarə sayı kəsr ola bilməz)
preds = preds.round().astype(int)

# Mənfi proqnozları 0-a çevir (məntiqsiz dəyərləri düzəlt)
preds = np.maximum(preds, 0)

print(f'Proqnoz statistikası:')
print(f'  Minimum : {preds.min()}')
print(f'  Maksimum: {preds.max()}')
print(f'  Ortalama: {preds.mean():.1f}')
print(f'  Sətir sayı: {len(preds)}')

In [ ]:
# Submission faylını yarat
# Kaggle formatı: datetime_id, count
sub['count'] = preds

# Faylı yadda saxla
sub.to_csv('submission.csv', index=False)
# index=False → sıra nömrəsini (0,1,2...) fayla yazma

print('✅ submission.csv hazırdır!')
print(f'Kaggle-a yükləmək üçün bu faylı istifadə et.')
print()
print('İlk 5 sətir:')
sub.head()

In [ ]:
# Proqnozların vizual yoxlaması
# Real train dağılımı ilə proqnoz dağılımını müqayisə et
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['count'], bins=40, color='steelblue', alpha=0.7,
             edgecolor='white', label='Train (real)')
axes[0].hist(preds, bins=40, color='coral', alpha=0.5,
             edgecolor='white', label='Proqnozlar')
axes[0].set_title('Real vs Proqnoz dağılımı')
axes[0].set_xlabel('Count')
axes[0].legend()

axes[1].plot(preds[:100], color='coral', linewidth=1.5, label='Proqnozlar (ilk 100)')
axes[1].set_title('Proqnozların zaman sırası (ilk 100)')
axes[1].set_xlabel('Sıra nömrəsi')
axes[1].set_ylabel('Proqnoz count')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Əgər iki histoqram oxşar formadadırsa — model yaxşı işləyir!')

## 🏁 Xülasə

| Addım | Nə etdik |
|---|---|
| 1 | Kitabxanaları yüklədik |
| 2 | train.csv, test.csv oxuduq |
| 3 | EDA: saatlar, mövsüm, korrelyasiya |
| 4 | 87 mənfi count-u sildik |
| 5 | hour, month, weekday, rush_hour, night əlavə etdik |
| 6 | log1p(count) ilə öyrətdik, Random Forest (200 ağac, CV=5) |
| 7 | submission.csv hazırladıq |

**Skoru yaxşılaşdırmaq üçün növbəti addımlar:**
- LightGBM modeli ilə dəyiş
- Optuna ilə hyperparameter tuning
- `casual` və `registered` ayrı-ayrı proqnozlaşdır